# Leveraging Databricks Lakehouse & system table to forecast your billing

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/uc/system_tables/uc-system-tables-flow.png?raw=true" width="800px" style="float: right">

As your billing information is saved in your system table, it's easy to leverage the lakehouse capabilities to forecast your consumption, and add alerts.

In this notebook, we'll run some analysis on the current consumption and build a Prophet model to forecast the future usage, based on SKU and workspace.

Because each workspace can have a different pattern / trend, we'll specialize multiple models on each SKU and Workspace and train them in parallel.

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=governance&org_id=984752964297111&notebook=%2F01-billing-tables%2F02-forecast-billing-tables&demo_name=uc-04-system-tables&event=VIEW&path=%2F_dbdemos%2Fgovernance%2Fuc-04-system-tables%2F01-billing-tables%2F02-forecast-billing-tables&version=1&user_hash=53e7df68e5fee236d97fc15226aeeed74331d6f7c2f836f9b915aecc5654a25a">

## Note: refresh your forecast data every day

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/uc/system_tables/uc-system-job.png?raw=true" style="float: left; margin: 20px" width="550px">

Make sure you refresh your forecast every day to get accurate previsions. 

To do that, simply click on Schedule and select "Every Day" to create a new Workflow refreshing this notebook on a daily basis. 

If you don't do it, your forecast will quickly expire and won't reflect potential consumption change / trigger alerts.

To make sure this happens, we also added a tracker in the Databricks SQL dashboard to display the number of days since the last forecast. 

## A note on pricing tables
Note that Pricing tables (containing the price information in `$` for each SKU) is available as a system table.

**Please consider these numbers as estimates which do not include any add-ons or discounts. It is using list price, not contractual. Please review your contract for more accurate information.**

## Granular Forecasts

SKUs are detailed per line of product and per region. To simplify our billing dashboard, we'll merge them under common SKU types i.e. grouping `STANDARD_ALL_PURPOSE_COMPUTE` and `PREMIUM_ALL_PURPOSE_COMPUTE` as `ALL_PURPOSE`

## Leveraging Databricks AI_FORECAST function

Databricks provides a built-in AI Forecast capability. See the [AI_FORECAST documentation](https://docs.databricks.com/en/sql/language-manual/functions/ai_forecast.html) for more details.

**Note that this might require the preview to be enabled to your workspace. If the preview isn't enable, we show you how to do the forecast below using prophet in python.**

**Make sure you run these next cells using a SQL WAREHOUSE as compute, not a classic cluster as the AI_FORECAST preview is only available in serverless for now.**

*Note: If the `AI_FORECAST` isn't yet available in your workspace, you can skip to the next section where we show you how to do the same in python.*

In [0]:
%sql
-- NOTE: make sure you run this notebook using a SQL Warehouse or Serverless endpoint (not a classic cluster).
SELECT assert_true(current_version().dbsql_version is not null, 'YOU MUST USE A SQL WAREHOUSE TO RUN THE NEXT CELLS HAVING THE AI_FORECAST FUNCTION, not a classic cluster');

In [0]:
%sql
select * from system.billing.usage u
  inner join system.billing.list_prices lp on u.cloud = lp.cloud and
    u.sku_name = lp.sku_name and
    u.usage_start_time >= lp.price_start_time and
    (u.usage_end_time <= lp.price_end_time or lp.price_end_time is null)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW data_to_predict AS (
WITH 
-- Classify SKU types and calculate usage metrics
classified_data AS (
    SELECT 
        u.workspace_id, 
        u.usage_date AS ds, 
        CASE
            WHEN u.sku_name LIKE '%ALL_PURPOSE%' THEN 'ALL_PURPOSE'
            WHEN u.sku_name LIKE '%JOBS%' THEN 'JOBS'
            WHEN u.sku_name LIKE '%SDP%' OR u.sku_name LIKE '%DLT%' THEN 'SDP'
            WHEN u.sku_name LIKE '%SQL%' THEN 'SQL'
            WHEN u.sku_name LIKE '%INFERENCE%' THEN 'MODEL_INFERENCE'
            ELSE 'OTHER'
        END AS sku,
        CAST(u.usage_quantity AS DOUBLE) AS dbus, 
        CAST(lp.pricing.default * u.usage_quantity AS DOUBLE) AS cost_at_list_price
    FROM 
        system.billing.usage u
    INNER JOIN 
        system.billing.list_prices lp 
    ON 
        u.cloud = lp.cloud 
        AND u.sku_name = lp.sku_name 
        AND u.usage_start_time >= lp.price_start_time 
        AND (u.usage_end_time <= lp.price_end_time OR lp.price_end_time IS NULL)
    WHERE 
        u.usage_unit = 'DBU'
),
-- Aggregate data by day, SKU, and workspace
daily_data AS (
    SELECT 
        ds,
        sku,
        workspace_id,
        SUM(dbus) AS dbus,
        SUM(cost_at_list_price) AS cost_at_list_price
    FROM 
        classified_data
    GROUP BY 
        ds, sku, workspace_id
),
-- Generate totals: workspace, SKU, and global
workspace_totals AS (
    SELECT ds, 'ALL' AS sku, workspace_id, SUM(dbus) AS dbus, SUM(cost_at_list_price) AS cost_at_list_price
    FROM daily_data
    GROUP BY ds, workspace_id
),
sku_totals AS (
    SELECT ds, sku, 'ALL' AS workspace_id, SUM(dbus) AS dbus, SUM(cost_at_list_price) AS cost_at_list_price
    FROM daily_data
    GROUP BY ds, sku
),
global_totals AS (
    SELECT ds, 'ALL' AS sku, 'ALL' AS workspace_id, SUM(dbus) AS dbus, SUM(cost_at_list_price) AS cost_at_list_price
    FROM daily_data
    GROUP BY ds
),
-- Filter for active workspaces
active_workspaces AS (
    SELECT DISTINCT workspace_id
    FROM daily_data
    WHERE ds >= CURRENT_DATE - INTERVAL 7 DAYS
),
-- Combine all data into a single dataset
combined_data AS (
    SELECT * FROM daily_data WHERE workspace_id IN (SELECT workspace_id FROM active_workspaces)
    UNION ALL
    SELECT * FROM workspace_totals WHERE workspace_id IN (SELECT workspace_id FROM active_workspaces)
    UNION ALL
    SELECT * FROM sku_totals
    UNION ALL
    SELECT * FROM global_totals
)
-- Add the MAX computation after the UNION (we use it as cap in our forecast)
SELECT 
    *,
    MAX(cost_at_list_price) OVER (PARTITION BY sku, workspace_id) AS max_cost_at_list_price
FROM combined_data
);

SELECT * FROM data_to_predict ORDER BY sku, workspace_id, ds limit 1000 ;

Let's now leverage the `AI_FORECAST` function to forecast the future pricing fur each sku/workspace id.

In [0]:
%sql
DROP TABLE IF EXISTS main.dbdemos_billing_forecast.billing_forecast;
CREATE TABLE main.dbdemos_billing_forecast.billing_forecast AS 
WITH data_to_predict_with_params AS (
  SELECT 
    '{"global_floor": 0, "min_changepoint_samples": 30, "global_cap": ' || (max_cost_at_list_price * 5) || '}' AS parameters,
    ds, 
    cost_at_list_price, 
    sku, 
    workspace_id
  FROM data_to_predict
)
SELECT 
  *, 
  current_date() as training_date
FROM ai_forecast(
  TABLE(data_to_predict_with_params),
  horizon => (SELECT MAX(ds) + INTERVAL 120 DAYS FROM data_to_predict),
  time_col => 'ds',
  value_col => 'cost_at_list_price',
  prediction_interval_width => 0.8,
  frequency => 'D',
  group_col => ARRAY('sku', 'workspace_id'),
  parameters => 'parameters'
);

In [0]:
%sql 
select * from main.dbdemos_billing_forecast.billing_forecast

In [0]:
%sql
DROP TABLE IF EXISTS main.dbdemos_billing_forecast.detailed_billing_forecast;
CREATE OR REPLACE TABLE main.dbdemos_billing_forecast.detailed_billing_forecast AS 
  WITH forecast_data as (
    SELECT 
      NULL as training_date, 
      ds, 
      sku, 
      workspace_id, 
      cost_at_list_price as past_list_cost, 
      NULL as cost_at_list_price_forecast, 
      NULL as cost_at_list_price_upper, 
      NULL as cost_at_list_price_lower 
      FROM data_to_predict 
        UNION 
    SELECT 
      training_date,
      ds, 
      sku, 
      workspace_id, 
      NULL as past_list_cost, 
      GREATEST(0, cost_at_list_price_forecast), 
      GREATEST(0, cost_at_list_price_upper), 
      GREATEST(0, cost_at_list_price_lower) 
      FROM  main.dbdemos_billing_forecast.billing_forecast)

    SELECT * EXCEPT(ds), 
      ds as date,
      past_list_cost is null as is_prediction,
      coalesce(past_list_cost, cost_at_list_price_forecast) as list_cost,
      avg(coalesce(past_list_cost, cost_at_list_price_forecast)) OVER (PARTITION BY sku, workspace_id ORDER BY ds ROWS BETWEEN 7 PRECEDING AND 7 FOLLOWING) AS list_cost_ma,
      avg(cost_at_list_price_forecast) OVER (PARTITION BY sku, workspace_id ORDER BY ds ROWS BETWEEN 7 PRECEDING AND 7 FOLLOWING) AS forecast_list_cost_ma, 
      avg(cost_at_list_price_lower) OVER (PARTITION BY sku, workspace_id ORDER BY ds ROWS BETWEEN 7 PRECEDING AND 7 FOLLOWING) AS forecast_list_cost_lower_ma,
      avg(cost_at_list_price_upper) OVER (PARTITION BY sku, workspace_id ORDER BY ds ROWS BETWEEN 7 PRECEDING AND 7 FOLLOWING) AS forecast_list_cost_upper_ma
    from forecast_data
    ORDER BY sku, workspace_id, ds;

SELECT * FROM main.dbdemos_billing_forecast.detailed_billing_forecast order by date desc limit 1000;

In [0]:
%sql
select * from main.dbdemos_billing_forecast.billing_forecast order by ds desc limit 100

## Manual AI Forecast leveraging Prophet and pandas UDF with spark

If the AI_FORECAST function isn't available yet in your workspace, you can do the prediction manually in python with prophet.

Open the [03-python-forecast-billing-tables]($./03-python-forecast-billing-tables) notebook for more details!


## Your forecast tables are ready for BI

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/uc/system_tables/dashboard-governance-billing.png?raw=true" width="500px" style="float: right">

dbdemos installed a dashboard for you to start exploring your data. 
<a dbdemos-dashboard-id="account-usage" href='/sql/dashboardsv3/01f0bfdb244218bbb334dcab244ece00'>Open the dashboard</a> to start exploring your billing data.<br/>
Remember that if you changed your Catalog and Schema, you'll have to update the queries.


## Exploring our forecasting data from the notebook

Our forecasting data is ready! We can explore it within the notebook directly, using Databricks built-in widget our any python plot library: